# Практическая работа № 4

In [110]:
import pandas as pd
from sklearn.linear_model import LinearRegression

from sklearnex import patch_sklearn
patch_sklearn()
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, \
    mean_absolute_percentage_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
import optuna

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)
C:\Users\79585\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Для начала загрузим таблицу с информацией о результатах построения моделей в практической работе № 3.

In [94]:
practice3_results = pd.read_csv("datasets/practice3_results.csv", index_col=0)

practice3_results

,Train Data,Train Data.1,Train Data.2,Train Data.3,Train Data.4,Test Data,Test Data.1,Test Data.2,Test Data.3,Test Data.4
NaN,R2,MSE,RMSE,MAE,MAPE,R2,MSE,RMSE,MAE,MAPE
Linear regression without regularization,0.82,0.184,0.4289,0.3427,1.7935,0.81,0.1876,0.4331,0.3452,3.3207
Linear regression with Lasso regularization,0.82,0.1841,0.4291,0.3432,1.7848,0.81,0.1874,0.4329,0.3455,3.3022
Linear regression with Ridge regularization,0.82,0.1843,0.4293,0.3427,1.7413,0.81,0.1873,0.4327,0.3441,3.2156
Linear regression with Elastic Net (get with GridSearch),0.82,0.1841,0.429,0.343,1.7771,0.81,0.1873,0.4328,0.3451,3.2871
Linear regression with Elastic Net (get with RandomSearch),0.82,0.1841,0.429,0.3427,1.7629,0.81,0.1873,0.4328,0.3445,3.2592
Linear regression with Elastic Net (get with Optuna),0.82,0.1843,0.4293,0.3429,1.7385,0.81,0.1872,0.4327,0.3442,3.2094
Polynomial regression (deg 2) without regularization,0.93,0.0725,0.2692,0.2042,0.9169,0.92,0.0786,0.2803,0.2118,1.71
Polynomial regression (deg 3) without regularization,0.97,0.0335,0.1832,0.1282,0.5769,0.96,0.0397,0.1991,0.1381,0.6519
Polynomial regression (deg 4) without regularization,0.98,0.018,0.1343,0.0921,0.3423,0.97,0.0345,0.1856,0.1252,0.8196


Удалим первую строку из датафрайма и сделаем корректные имена столбцов

In [96]:
practice3_results.drop(practice3_results.index[0], axis=0, inplace=True)

metrics = ["R2", "MSE", "RMSE", "MAE", "MAPE"]

columns = pd.MultiIndex.from_product([["Train Data", "Test Data"], metrics])
practice3_results.columns = columns
practice3_results

Train Data                  \
                                                           R2     MSE    RMSE   
Linear regression without regularization                 0.82   0.184  0.4289   
Linear regression with Lasso regularization              0.82  0.1841  0.4291   
Linear regression with Ridge regularization              0.82  0.1843  0.4293   
Linear regression with Elastic Net (get with Gr...       0.82  0.1841   0.429   
Linear regression with Elastic Net (get with Ra...       0.82  0.1841   0.429   
Linear regression with Elastic Net (get with Op...       0.82  0.1843  0.4293   
Polynomial regression (deg 2) without regulariz...       0.93  0.0725  0.2692   
Polynomial regression (deg 3) without regulariz...       0.97  0.0335  0.1832   
Polynomial regression (deg 4) without regulariz...       0.98   0.018  0.1343   
Polynomial regression(deg 2) with L1 (get with ...       0.93  0.0727  0.2697   
Polynomial regression(deg 2) with L1 (get with ...       0.91  0.0921  0.3035   
Polynomial regression(deg 2) with L1 (get with ...       0.93  0.0727  0.2697   
Polynomial regression (deg 2) with L2 (get with...       0.93  0.0725  0.2692   
Polynomial regression (deg 2) with L2 (get with...       0.93  0.0725  0.2692   
Polynomial regression (deg 2) with L2 (get with...       0.93  0.0725  0.2692   
Polynomial regression (deg 2) with Elastic Net ...       0.93  0.0726  0.2695   
Polynomial regression (deg 2) with Elastic Net ...       0.93  0.0726  0.2695   
Polynomial regression (deg 2) with Elastic Net ...       0.93  0.0725  0.2692   
Polynomial regression (deg 3) with L1 (GridSearch)       0.97  0.0349  0.1869   
Polynomial regression (deg 3) with L1 (RandomSe...       0.92  0.0814  0.2853   
Polynomial regression (deg 3) with L1 (Optuna)           0.97  0.0349  0.1869   
Polynomial regression (deg 3) with L2 (GridSearch)       0.97  0.0336  0.1832   
Polynomial regression (deg 3) with L2 (RandomSe...       0.97  0.0335  0.1832   
Polynomial regression (deg 3) with L2 (Optuna)           0.97  0.0335  0.1832   
Polynomial regression (deg 3) with Elastic Net ...       0.97  0.0344  0.1854   
Polynomial regression (deg 3) with Elastic Net ...       0.97  0.0345  0.1858   
Polynomial regression (deg 3) with Elastic Net ...       0.97  0.0339  0.1841   
Polynomial regression (deg 4) with L1 (GridSearch)       0.98  0.0231  0.1519   
Polynomial regression (deg 4) with L1 (RandomSe...       0.91    0.09     0.3   
Polynomial regression (deg 4) with L1 (Optuna)           0.98  0.0231  0.1519   
Polynomial regression (deg 4) with L2 (GridSearch)       0.98  0.0181  0.1346   
Polynomial regression (deg 4) with L2 (RandomSe...       0.98  0.0182   0.135   
Polynomial regression (deg 4) with L2 (Optuna)           0.98   0.018  0.1343   
Polynomial regression (deg 4) with Elastic Net ...       0.98  0.0225  0.1501   
Polynomial regression (deg 4) with Elastic Net ...       0.98  0.0209  0.1445   
Polynomial regression (deg 4) with Elastic Net ...       0.98  0.0208  0.1442   

                                                                   Test Data  \
                                                       MAE    MAPE        R2   
Linear regression without regularization            0.3427  1.7935      0.81   
Linear regression with Lasso regularization         0.3432  1.7848      0.81   
Linear regression with Ridge regularization         0.3427  1.7413      0.81   
Linear regression with Elastic Net (get with Gr...   0.343  1.7771      0.81   
Linear regression with Elastic Net (get with Ra...  0.3427  1.7629      0.81   
Linear regression with Elastic Net (get with Op...  0.3429  1.7385      0.81   
Polynomial regression (deg 2) without regulariz...  0.2042  0.9169      0.92   
Polynomial regression (deg 3) without regulariz...  0.1282  0.5769      0.96   
Polynomial regression (deg 4) without regulariz...  0.0921  0.3423      0.97   
Polynomial regression(deg 2) with L1 (get with ...  0.2046  0.9051      0.92   
Polynomial r

Создадим две таблицы для работы: первая - с тренировочными и валидационными выборками, а вторая - с тестовыми данными. В итоге работы объединим их в финальную результирующую таблицу

In [107]:
train_and_val_metrics_comparing_columns = pd.MultiIndex.from_product([["Train Data", "Val Data"], metrics]).to_frame()
train_and_val_metrics_comparing = pd.DataFrame(columns=train_and_val_metrics_comparing_columns)
train_and_val_metrics_comparing

,"(Train Data, R2)","(Train Data, MSE)","(Train Data, RMSE)","(Train Data, MAE)","(Train Data, MAPE)","(Val Data, R2)","(Val Data, MSE)","(Val Data, RMSE)","(Val Data, MAE)","(Val Data, MAPE)"


In [109]:
test_metrics_columns = pd.MultiIndex.from_product([["Test Data"], metrics])
test_metrics_info = pd.DataFrame(columns=test_metrics_columns)
test_metrics_info

Empty DataFrame
Columns: [(Test Data, R2), (Test Data, MSE), (Test Data, RMSE), (Test Data, MAE), (Test Data, MAPE)]
Index: []

Для избежания дублирования кода используем некоторые кастомные функции

Функция для получения метрик `get_metrics`

In [ ]:
def get_metrics(y_test, y_predicted, print_results=False):
    names_of_metrics = ["R2", "MSE", "RMSE", "MAE", "MAPE"]
    results = [
        round(r2_score(y_test, y_predicted), 2),
        round(mean_squared_error(y_test, y_predicted), 4),
        round(root_mean_squared_error(y_test, y_predicted), 4),
        round(mean_absolute_error(y_test, y_predicted), 4),
        round(mean_absolute_percentage_error(y_test, y_predicted), 4)
    ]

    if print_results:
        for name, result in zip(names_of_metrics, results):
            print(f"{name}: {result}")

    return results

Функция для вставки метрик в таблицу с тренировочными и валидационными данными `input_metrics_into_the_train_val_table`

In [ ]:
def input_metrics_into_the_train_val_table(name_of_model,
                          predict_on_train_y, y_train,
                          predict_on_val_y, y_val):
    val_metrics = get_metrics(y_val, predict_on_val_y)

    train_metrics = get_metrics(y_train, predict_on_train_y)

    train_and_val_metrics_comparing.loc[name_of_model] = train_metrics  + val_metrics

Функция для вставки метрик в таблицу с тестовыми данными `input_metrics_into_the_train_val_table`

In [ ]:
def input_metrics_into_the_train_val_table(name_of_model,
                          predict_on_test_y, y_test):
    test_metrics = get_metrics(y_test, predict_on_test_y)

    test_metrics_info.loc[name_of_model] = test_metrics

Функция `train_models_with_reg ` используется для получения моделей с L1, L2 и ElasticNet регуляризациями с помощью GridSearch, RandomizedSearchCV и фреймворка Optuna.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
def train_models_with_reg(X_train, y_train, X_val, y_val, reg_config):

    result={}


    for name, reg_info in reg_config.items():

        param_grid = reg_info["param_grid"]
        type_of_reg = reg_info["reg"]

        grid_search_model = GridSearchCV(
            param_grid=param_grid,
            estimator=type_of_reg(max_iter=1000),
            scoring="neg_mean_squared_error",
            cv=3
        )

        grid_search_model.fit(X_train, y_train)

        random_search_model = RandomizedSearchCV(
            param_distributions=param_grid,
            estimator=type_of_reg(max_iter=1000),
            scoring="neg_mean_squared_error",
            random_state=42,
            cv=3
        )

        random_search_model.fit(X_train, y_train)

        def objective(trial):
            trial_params = {}
            for p_name, p_values in param_grid.items():
                if isinstance(p_values[0], float):
                    trial_params[p_name] = trial.suggest_float(p_name, min(p_values), max(p_values), log=True)
                else:
                    trial_params[p_name] = trial.suggest_categorical(p_name, p_values)

            model = type_of_reg(**trial_params)
            model.fit(X_train, y_train)
            score = mean_squared_error(y_val, model.predict(X_val))
            return score

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=100, show_progress_bar=False)

        optuna_model = type_of_reg(**study.best_params)
        optuna_model.fit(X_train, y_train)

        result[name] = {}
        result[name]["GridSearch"] = {
            "model": grid_search_model.best_estimator_,
            "best_params": grid_search_model.best_params_
        }
        result[name]["RandomSearch"] = {
            "model": random_search_model.best_estimator_,
            "best_params": random_search_model.best_params_
        }
        result[name]["Optuna"] = {
            "model": optuna_model,
            "best_params": study.best_params
        }

    return result

## Разделение данных на тренировочную, валидационную и тестовую выборки

In [ ]:


X_train, X_val_test, y_train, y_val_test = train_test_split(X)

## Обучение моделей с сохранением данных в локальные переменные

### Модель множественной регрессии без регуляризации

In [ ]:
multiple_regression = LinearRegression()
multiple_regression.fit()